# Week 1 · Notebook 1 — The data, and your first tool

Welcome. Over three weeks you'll build a real algorithmic trading system: it reads
the market, forms strategies, tests them honestly, and — by week 3 — *learns* to
trade. Everything compounds; nothing you build gets thrown away.

Today is the gentlest day on purpose. Two goals:
1. Understand **the environment** — the world your strategies will live in.
2. Build your **first tool**: a price-charting function you'll reuse all week.

If your setup works and you draw a chart by the end, today succeeded.

## 1. The environment: what your strategies see

Every strategy in this program acts inside one shared world — a `DataFeed`. Think
of it as the market, frozen into a table you can query. It holds, for a whole
universe of stocks aligned to the same calendar:

- `feed.close` — closing prices, shape **(days × stocks)**
- `feed.returns` — daily returns (how much each stock moved), same shape
- `feed.volume` — how much traded

This is the single source of truth. Indicators read from it, strategies act on it,
the backtester walks through it day by day, and in week 3 the reinforcement-learning
agent will live inside it too. Learn its shape now and everything later is familiar.

In [4]:
import sys, os
while not os.path.isdir('src') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
sys.path.insert(0, 'src')
import numpy as np
import matplotlib.pyplot as plt

from tradinglab.data_feed import DataFeed

feed = DataFeed.from_dir('data/egx')
print('universe :', feed.symbols)
print('days     :', feed.n_days)
print('close is a table of shape (days, stocks):', feed.close.shape)
print()
print('last 5 closing prices of', feed.symbols[0], ':', feed.close[-5:, 0].round(2))
print('its last 5 daily returns             :', feed.returns[-5:, 0].round(4))

RuntimeError: NumPy was built with baseline optimizations: 
(X86_V2) but your machine doesn't support:
(X86_V2).

## 2. Your first tool — `plot_price`

You'll look at prices constantly, so build a reusable chart function. Given the
figure is already set up for you, fill in the drawing: plot the closing price, then
plot each optional overlay (like a moving average) on top.

**In:** `dates`, `close`, and an optional `overlays` dict like `{'SMA20': array}`.
**Out:** the chart's `ax` (already returned for you).
**Hint:** `ax.plot(dates, close, label='close')`, then loop over `overlays.items()`.
**Done when:** the check counts the right number of lines.

In [3]:
def plot_price(dates, close, overlays=None, title='Price'):
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.plot(dates, close, label='close', linewidth=1.2)
    for name, series in (overlays or {}).items():
        ax.plot(dates, series, label=name, linewidth=1.0)
    ax.set_title(title); ax.legend(); ax.grid(alpha=0.3)
    return ax

ax = plot_price(feed.dates, feed.close[:, 0],
                overlays={'flat line': np.full(feed.n_days, feed.close[:,0].mean())},
                title=feed.symbols[0])
assert len(ax.lines) == 2, 'expected close + 1 overlay'
plt.show()
print('plot_price works ✓ — this is your chart tool for the whole week')

NameError: name 'feed' is not defined

## 3. What you can do this week

Now that you can see the data, here's the arc:
- **Notebook 2** — build the indicators (moving average, RSI) that turn prices into
  signals.
- **Notebook 3** — turn signals into a strategy, and write the one function at the
  heart of the whole engine.
- **Notebook 4** — backtest your strategy and **report** the result honestly against
  the benchmark.
- **Notebook 5** — put it all on a dashboard.

**Graduate** `plot_price` into `src/tradinglab/charting.py` (replace the stub), then:
`uv run pytest week1/tests/`. Tomorrow you build the indicators that go on this
chart.

## 4. Start with 1,000 EGP — portfolio vs EGX 30

To make the result concrete, we invest equally across all stocks in the feed and hold
that portfolio throughout the available period. The portfolio starts with **1,000 EGP**.
We compare its final value, profit, and maximum drawdown with the real EGX 30 index.

The simulator applies weights chosen on day `t` to the return from `t` to `t+1`, so the
comparison avoids using a closing price before it was known.

In [ ]:
from tradinglab.simulator import PortfolioSimulator
from tradinglab import metrics

START_CAPITAL = 1000.0

# Equal-weight buy-and-hold: every stock has the same allocation.
weights = np.full((feed.n_days, feed.n_assets), 1.0 / feed.n_assets)
simulator = PortfolioSimulator(feed, benchmark='egx30')
result = simulator.run(weights, start=0, end=feed.n_days - 1)

portfolio_value = result['portfolio'] * START_CAPITAL
benchmark_value = result['benchmark'] * START_CAPITAL
portfolio_profit = portfolio_value[-1] - START_CAPITAL
benchmark_profit = benchmark_value[-1] - START_CAPITAL
portfolio_return = metrics.total_return(result['portfolio_returns'])
benchmark_return = metrics.total_return(result['benchmark_returns'])
portfolio_drawdown = metrics.max_drawdown(result['portfolio_returns'])
benchmark_drawdown = metrics.max_drawdown(result['benchmark_returns'])

print(f'Initial capital       : {START_CAPITAL:,.2f} EGP')
print(f'Portfolio final value : {portfolio_value[-1]:,.2f} EGP')
print(f'Portfolio profit      : {portfolio_profit:+,.2f} EGP ({portfolio_return:+.2%})')
print(f'Portfolio max drawdown: {portfolio_drawdown:.2%}')
print()
print(f'EGX 30 final value    : {benchmark_value[-1]:,.2f} EGP')
print(f'EGX 30 profit         : {benchmark_profit:+,.2f} EGP ({benchmark_return:+.2%})')
print(f'EGX 30 max drawdown   : {benchmark_drawdown:.2%}')
print()
print(f'Outperformance vs EGX 30: {portfolio_profit - benchmark_profit:+,.2f} EGP')
print(f'Beat EGX 30           : {"YES" if portfolio_value[-1] > benchmark_value[-1] else "NO"}')

plt.figure(figsize=(11, 5))
plt.plot(result['dates'], portfolio_value, label='Equal-weight portfolio', linewidth=1.5)
plt.plot(result['dates'], benchmark_value, label='EGX 30', linewidth=1.5)
plt.axhline(START_CAPITAL, color='black', linewidth=0.8, alpha=0.5)
plt.title('1,000 EGP: Portfolio vs EGX 30')
plt.ylabel('Value (EGP)')
plt.legend()
plt.grid(alpha=0.3)
plt.show()